In [ ]:
# ── Standard library ───────────────────────────────────────────────────────────
import os
import math
import textwrap
# import pdb  
from statistics import mean
# ── Data & preprocessing ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
# ── Statistics ────────────────────────────────────────────────────────────────
from scipy import stats  
from scipy.stats import ( 
    ttest_ind,
    spearmanr,
    pearsonr,
    chi2,
    zscore,
    shapiro,
    normaltest,
    levene, ttest_ind, 
    mannwhitneyu
)
# ── Modeling ─────────────────────────────────────────────────────────────────
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pymer4.models import Lmer 

# ── Plotting ──────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches  # custom legend patches
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# Define the directory where the files are saved
directory = 'data'

# Define file paths for each CSV file
file_paths = {
    "relativeShift_OCD": os.path.join(directory, 'CrypticCreaturesBayesianLearner_relativeShift_OCD.csv'),
    "relativeShift_controls": os.path.join(directory, 'CrypticCreaturesBayesianLearner_relativeShift_controls.csv'),
    "relativeED_OCD": os.path.join(directory, 'CrypticCreaturesBayesianLearner_relativeED_OCD.csv'),
    "relativeED_controls": os.path.join(directory, 'CrypticCreaturesBayesianLearner_relativeED_controls.csv'),
    "relativeID_OCD": os.path.join(directory, 'CrypticCreaturesBayesianLearner_relativeID_OCD.csv'),
    "relativeID_controls": os.path.join(directory, 'CrypticCreaturesBayesianLearner_relativeID_controls.csv'),
    "CrypticCreatures": os.path.join(directory, 'CrypticCreatures_BayesianLearner.csv')
}

# Read each CSV file into a dictionary of DataFrames
dataframes = {}
for name, path in file_paths.items():
    if os.path.exists(path):
        dataframes[name] = pd.read_csv(path)
    else:
        print(f"Error: File not found - {path}")
    
CrypticCreatures_adapted = dataframes["CrypticCreatures"]

Data prep

In [ ]:
# Sort the DataFrame 
CrypticCreatures_adapted = CrypticCreatures_adapted.sort_values(by=['id', 'task_id', 'run', 'trial']).reset_index(drop=True)
CrypticCreatures_adapted['total_iq'] = CrypticCreatures_adapted['icar_totalscore']

CrypticCreatures_adapted['task_break'] = ((CrypticCreatures_adapted['trial'] == 1) & (CrypticCreatures_adapted['run'] == 1)).astype(int)
CrypticCreatures_adapted['block_break'] = ((CrypticCreatures_adapted['trial'] == 1) & (CrypticCreatures_adapted['run'] == 2)).astype(int)

CrypticCreatures_adapted['global_trial'] = 0
CrypticCreatures_adapted['task_trial'] = 0

# Assign global trials and task-specific trials
for participant_id in CrypticCreatures_adapted['id'].unique():
    participant_data = CrypticCreatures_adapted[CrypticCreatures_adapted['id'] == participant_id]
    
    
    global_trial_counter = 1
    
    for task in participant_data['task_id'].unique():
        task_data = participant_data[participant_data['task_id'] == task]
        task_trial_counter = 1  
        
        for index in task_data.index:
            CrypticCreatures_adapted.at[index, 'global_trial'] = global_trial_counter
            global_trial_counter += 1
            
            CrypticCreatures_adapted.at[index, 'task_trial'] = task_trial_counter
            task_trial_counter += 1

grouped = CrypticCreatures_adapted.groupby(["id", "task_id", "run"])["trial"].nunique()

df_check = grouped.unstack(level=["task_id", "run"]).fillna(0)
 
ocd_color = '#5f8cff'
control_color = '#b4a854'

# Define colors for all the plots
colors_palette = [
    '#7c242b',  # Soft Purple
    '#a37c64',  # Warm Brown
    '#a794ba',  # Muted Pink
    '#8a9f6a',  # Muted Green
    '#859ba3',  # Stone Gray
    '#9a8b4e',  # Olive Green
    '#7392c9',   # Slate Blue
    '#c64865',  # pink
    '#ec6a35'  # orange
]

In [ ]:
# Compute average accuracy and confidence per id and task_id
avg_metrics_per_task = CrypticCreatures_adapted.groupby(['id', 'task_id']).agg(
    avg_accuracy=('chosen_outcome', 'mean'),
    avg_confidence=('confidence', 'mean'),
    avg_RT=('RT_log', 'mean')
).reset_index()

# Compute overall average accuracy and confidence per id across all task_ids
avg_metrics_overall = CrypticCreatures_adapted.groupby('id').agg(
    overall_avg_accuracy=('chosen_outcome', 'mean'),
    overall_avg_confidence=('confidence', 'mean'),
    overall_avg_RT=('RT_log', 'mean'),
    overall_av_repetErr=('repet_err_ed', 'mean'),
    overall_diff_BLR=('signed_confidence_deviation', 'mean'),
    overall_diff_sumprior=('signed_prior_deviation', 'mean'),
).reset_index()

# Merge back
CrypticCreatures_adapted = CrypticCreatures_adapted.merge(avg_metrics_per_task, on=['id', 'task_id'], how='left', suffixes=('', '_per_task'))
# Merge back
CrypticCreatures_adapted = CrypticCreatures_adapted.merge(avg_metrics_overall, on='id', how='left', suffixes=('', '_overall'))


Some correlations

In [ ]:
participant_id_col = 'id'                        
gc_col = 'prev_BLR_confidence'         
cc_col = 'sum_prior_chosen_features'    

required_cols = [participant_id_col, gc_col, cc_col]
missing_cols = [col for col in required_cols if col not in CrypticCreatures_adapted.columns]
if missing_cols:
    raise ValueError(f"ERROR: The following required columns are missing from the DataFrame 'data': {missing_cols}")

print(f"Calculating Spearman correlations between '{gc_col}' and '{cc_col}'...")

# --- 1. Overall Correlation (Across all trials and people) ---
print("\n--- Overall Correlation ---")
# Drop rows with NaNs in either column before calculating
data_subset_overall = CrypticCreatures_adapted[[gc_col, cc_col]].dropna()

if len(data_subset_overall) < 2:
    print("Not enough non-NaN data points to calculate overall correlation.")
else:
    try:
        overall_rho, overall_p_value = stats.spearmanr(data_subset_overall[gc_col], data_subset_overall[cc_col])
        print(f"Spearman Correlation across all trials: rho = {overall_rho:.3f}")
        print(f"P-value: p = {overall_p_value:.4g}") 
    except ValueError as e:
        print(f"Could not calculate overall correlation. Error: {e}")

# --- 2. Average Within-Participant Correlation ---
def calculate_within_subject_spearman_corr(group):
    """Calculates Spearman correlation for a single participant's data."""
    group_subset = group[[gc_col, cc_col]].dropna()
    if len(group_subset) < 2 or group_subset[gc_col].nunique(dropna=False) <= 1 or group_subset[cc_col].nunique(dropna=False) <= 1:
        return np.nan # Not enough data or no variance
    try:
        rho, _ = stats.spearmanr(group_subset[gc_col], group_subset[cc_col])
        if np.isnan(rho):
             return np.nan
        return rho
    except ValueError:
        return np.nan

within_subject_rhos = CrypticCreatures_adapted.groupby(participant_id_col).apply(calculate_within_subject_spearman_corr)
valid_rhos = within_subject_rhos.dropna()

if len(valid_rhos) == 0:
    print("Could not calculate correlation for any participant.")
elif len(valid_rhos) < len(within_subject_rhos):
    print(f"Correlation calculated for {len(valid_rhos)} out of {len(within_subject_rhos)} participants.")
else:
    print(f"Correlation calculated for all {len(valid_rhos)} participants.")

if len(valid_rhos) > 0:
    mean_rho_simple = valid_rhos.mean()
    std_rho = valid_rhos.std()
    print(f"\nSimple Average of Spearman correlations: Mean rho = {mean_rho_simple:.3f}, SD = {std_rho:.3f}")
    valid_rhos_clipped = valid_rhos.clip(-0.999999, 0.999999) 
    z_transformed_rhos = np.arctanh(valid_rhos_clipped)
    mean_z = z_transformed_rhos.mean()
    mean_rho_fisher = np.tanh(mean_z)
    sd_z = z_transformed_rhos.std()

    print(f"Average Spearman Correlation using Fisher's z-transform: Mean rho = {mean_rho_fisher:.3f}")
    # print(f"  (Based on Mean z = {mean_z:.3f}, SD z = {sd_z:.3f})") # Optional detail
    print(f"Standard Deviation of individual participant correlations (rho values): SD = {std_rho:.3f}")

Mixed models task variables & Bayesian Learner

In [ ]:
# ===========================================
# 1) Helpers
# ===========================================
def zscore_within(x, ddof=1):
    m = x.mean()
    s = x.std(ddof=ddof)
    return (x - m) / s if s != 0 else x - m

# ===========================================
# 2) Trial-level within-id z-scores (.sc)
# ===========================================
trial_vars = [
    'confidence',
    'sum_prior_chosen_features',
    'BLR_confidence',
    'prev_BLR_confidence',
    'entropy',
    'RT_log',
]

for var in trial_vars:
    if var in CrypticCreatures_adapted.columns:
        CrypticCreatures_adapted[f'{var}.sc'] = (
            CrypticCreatures_adapted
            .groupby('id')[var]
            .transform(lambda v: zscore_within(v, ddof=1))
        )

# ===========================================
# 3) Participant-level global z-scores
# ===========================================
if 'age' in CrypticCreatures_adapted.columns:
    CrypticCreatures_adapted['age.sc'] = (
        (CrypticCreatures_adapted['age'] - CrypticCreatures_adapted['age'].mean())
        / CrypticCreatures_adapted['age'].std(ddof=1)
    )

# IQ: use icar_totalscore → total_iq.sc (consistent with your example style)
if 'icar_totalscore' in CrypticCreatures_adapted.columns:
    CrypticCreatures_adapted['total_iq.sc'] = (
        (CrypticCreatures_adapted['icar_totalscore'] - CrypticCreatures_adapted['icar_totalscore'].mean())
        / CrypticCreatures_adapted['icar_totalscore'].std(ddof=1)
    )

# ===========================================
# 4) Ensure categorical variables are 'category'
# ===========================================
cat_cols = ['id', 'patientstatus', 'task_id', 'gender', 'chosen_outcome_shift', 'chosen_outcome']
for col in cat_cols:
    if col in CrypticCreatures_adapted.columns:
        CrypticCreatures_adapted[col] = CrypticCreatures_adapted[col].astype(str).astype('category')

# Remove any unused categories
for col in [c for c in cat_cols if c in CrypticCreatures_adapted.columns]:
    CrypticCreatures_adapted[col] = CrypticCreatures_adapted[col].cat.remove_unused_categories()

# ===========================================
# 5) Build factors dict from observed levels 
# ===========================================
factors = {}
for col in ['patientstatus', 'chosen_outcome_shift', 'task_id', 'gender']:
    if col in CrypticCreatures_adapted.columns and hasattr(CrypticCreatures_adapted[col], 'cat'):
        factors[col] = list(CrypticCreatures_adapted[col].cat.categories)

# Quick sanity printout
for k in ['patientstatus', 'task_id', 'gender']:
    if k in factors:
        print(f"{k} levels:", factors[k])
print("Factors dictionary:", factors)


In [ ]:
# Mixed models
for col in ['patientstatus', 'gender', 'task_id', 'id']:
    if col in CrypticCreatures_adapted.columns:
        CrypticCreatures_adapted[col] = CrypticCreatures_adapted[col].astype('category')

factors = {
    col: list(CrypticCreatures_adapted[col].cat.categories)
    for col in ['patientstatus', 'gender', 'task_id']
    if col in CrypticCreatures_adapted.columns
    and hasattr(CrypticCreatures_adapted[col], 'cat')
}

def drop_na_for(df, cols):
    return df.dropna(subset=[c for c in cols if c in df.columns])

base_covars = ['age.sc', 'gender', 'total_iq.sc', 'patientstatus', 'task_id']

data_cc = drop_na_for(
    CrypticCreatures_adapted,
    ['confidence', 'sum_prior_chosen_features.sc'] + base_covars,
)
data_gc = drop_na_for(
    CrypticCreatures_adapted,
    ['confidence', 'prev_BLR_confidence.sc'] + base_covars,
)
data_joint = drop_na_for(
    CrypticCreatures_adapted,
    ['confidence', 'sum_prior_chosen_features.sc', 'prev_BLR_confidence.sc'] + base_covars,
)

print(f"N: CC={len(data_cc)}, GC={len(data_gc)}, joint={len(data_joint)}")

modelsConf = {}

# CC-only
modelsConf["CC_PT_RI"] = Lmer(
    "confidence ~ patientstatus * sum_prior_chosen_features.sc "
    "+ task_id + age.sc + gender + total_iq.sc "
    "+ (1 + task_id | id) + (1 | task_id)",
    data=data_cc,
)
modelsConf["CC_PT_RI"].fit(factors=factors, old_optimizer=True)

# CC RS
modelsConf["CC_PT_RS"] = Lmer(
    "confidence ~ patientstatus * sum_prior_chosen_features.sc "
    "+ age.sc + gender + total_iq.sc "
    "+ (1 + sum_prior_chosen_features.sc + task_id | id) + (1 | task_id)",
    data=data_cc,
)
modelsConf["CC_PT_RS"].fit(factors=factors, old_optimizer=True)

print("\nCC-only:")
print(modelsConf["CC_PT_RI"].summary())
print(modelsConf["CC_PT_RS"].summary())

# GC-only
modelsConf["GC_PT_RI"] = Lmer(
    "confidence ~ patientstatus * prev_BLR_confidence.sc "
    "+ task_id + age.sc + gender + total_iq.sc "
    "+ (1 + task_id | id) + (1 | task_id)",
    data=data_gc,
)
modelsConf["GC_PT_RI"].fit(factors=factors, old_optimizer=True)

# GC RS - structure ensures convergence
modelsConf["GC_PT_RS"] = Lmer(
    "confidence ~ patientstatus * prev_BLR_confidence.sc "
    "+ task_id + age.sc + gender + total_iq.sc "
    "+ (1 + prev_BLR_confidence.sc | id) + (1 | task_id)",
    data=data_gc,
)
modelsConf["GC_PT_RS"].fit(factors=factors, old_optimizer=True)

print("\nGC-only:")
print(modelsConf["GC_PT_RI"].summary())
print(modelsConf["GC_PT_RS"].summary())

modelsConf["All_PT_RS_both"] = Lmer(
    "confidence ~ patientstatus * (sum_prior_chosen_features.sc + prev_BLR_confidence.sc) "
    "+ age.sc + gender + total_iq.sc "
    "+ (1 + sum_prior_chosen_features.sc + prev_BLR_confidence.sc | id) + (1 | task_id)",
    data=data_joint,
)
modelsConf["All_PT_RS_both"].fit(factors=factors, old_optimizer=True)

print("\nJoint:")
print(modelsConf["All_PT_RS_both"].summary())

print("\nFit stats:")
for name in ["CC_PT_RI", "CC_PT_RS", "GC_PT_RI", "GC_PT_RS", "All_PT_RS_both"]:
    m = modelsConf[name]
    print(f"  {name}: AIC={m.AIC:.1f}, BIC={m.BIC:.1f}, LL={m.logLike:.1f}")


In [ ]:
# --- compute per-id Spearman correlations ---
per_id_rows = []
for pid, g in CrypticCreatures_adapted[['id', 'sum_prior_chosen_features', 'prev_BLR_confidence' ]].dropna().groupby('id'):
    x = g['sum_prior_chosen_features'].to_numpy()
    y = g['prev_BLR_confidence' ].to_numpy()
    if len(x) >= 3 and np.unique(x).size > 1 and np.unique(y).size > 1:
        rho, p = spearmanr(x, y)
        per_id_rows.append({'id': pid, 'rho': rho})
    else:
        continue

per_id_corr = pd.DataFrame(per_id_rows)

if per_id_corr.empty:
    raise ValueError("No valid per-id correlations could be computed. "
                     "Check for missing data or zero variance within IDs.")

# --- summarize (simple mean/SD across IDs) ---
mean_rho = per_id_corr['rho'].mean()
sd_rho   = per_id_corr['rho'].std(ddof=1)
n_ids    = per_id_corr.shape[0]
z_vals = np.arctanh(per_id_corr['rho'].clip(-0.999999, 0.999999))
mean_rho_fisher = np.tanh(z_vals.mean())
sd_z            = z_vals.std(ddof=1)

print(f"N IDs used = {n_ids}")
print(f"Per-ID Spearman ρ: mean = {mean_rho:.3f}, SD = {sd_rho:.3f}")
print(f"Per-ID Spearman ρ (Fisher-z mean): mean = {mean_rho_fisher:.3f}  [SD(z) = {sd_z:.3f}]")


In [ ]:
# ============================================================
# 0) Figure 4B
use_patientstatus = True

dot_color = '#57504d'  # your color

if 'modelsConf' not in globals():
    raise NameError("modelsConf not found. Fit models first so modelsConf[...] exists.")

pt_keys   = [k for k in modelsConf if k.startswith("All_PT_")]
nopt_keys = [k for k in modelsConf if k.startswith("All_NoPT_")]

if not pt_keys and not nopt_keys:
    raise ValueError("No 'All_PT_*' or 'All_NoPT_*' models found in modelsConf.")

def _best_by_bic(keys):
    ks = [k for k in keys if hasattr(modelsConf[k], 'BIC')]
    if not ks:
        raise ValueError("No models with BIC in the provided key set.")
    return min(ks, key=lambda k: modelsConf[k].BIC)

winner_key = _best_by_bic(pt_keys) if use_patientstatus else _best_by_bic(nopt_keys)
winner     = modelsConf[winner_key]

print(f"Using winning All model: {winner_key} | "
      f"AIC={winner.AIC:.3f}, BIC={winner.BIC:.3f}, LogLik={winner.logLike:.3f}")

coefs = winner.coefs.copy()

drop_patterns = ['(Intercept)', 'Intercept', 'age.sc', 'total_iq.sc', 'task_id', 'gender']
mask_drop = pd.Series(False, index=coefs.index)
for pat in drop_patterns:
    mask_drop = mask_drop | coefs.index.str.contains(rf'^{pat}$') | coefs.index.str.contains(pat)

coefs_plot = coefs[~mask_drop].copy()

rename_map = {
    'patientstatus1': 'Group',
    'sum_prior_chosen_features.sc': 'CC',
    'prev_BLR_confidence.sc': 'GC',
    'patientstatus1:sum_prior_chosen_features.sc': 'Group × CC',
    'patientstatus1:prev_BLR_confidence.sc': 'Group × GC',
    'sum_prior_chosen_features': 'CC',
    'prev_BLR_confidence': 'GC',
    'patientstatus1:sum_prior_chosen_features': 'Group × CC',
    'patientstatus1:prev_BLR_confidence': 'Group × GC',
}

pretty_labels = []
for term in coefs_plot.index:
    if term in rename_map:
        pretty_labels.append(rename_map[term])
    elif ':' in term:
        # general fallback for interactions
        parts = term.split(':')
        pretty_labels.append(" ×\n".join(rename_map.get(p, p) for p in parts))
    else:
        pretty_labels.append(rename_map.get(term, term))
coefs_plot['Pretty'] = pretty_labels

desired_order = [
    'Group', 'CC', 'GC', 'Group × CC', 'Group × GC'
]
order_idx = []
for lab in desired_order:
    if lab in coefs_plot['Pretty'].values:
        order_idx.extend(coefs_plot.index[coefs_plot['Pretty'] == lab].tolist())
order_idx += [ix for ix in coefs_plot.index if ix not in order_idx]
coefs_plot = coefs_plot.loc[order_idx]

print("\nKey coefficients (3 d.p.):")
show_terms = ['patientstatus1',
              'sum_prior_chosen_features.sc',
              'prev_BLR_confidence.sc',
              'patientstatus1:sum_prior_chosen_features.sc',
              'patientstatus1:prev_BLR_confidence.sc']
show_terms = [t for t in show_terms if t in coefs.index]
if show_terms:
    tbl = (coefs.loc[show_terms, ['Estimate','SE','P-val']]
                 .round(3)
                 .rename(index=rename_map))
    print(tbl.to_string())
else:
    print("(No patientstatus terms found; showing available non-demographic terms.)")
    print(coefs_plot[['Estimate','SE','P-val']].round(3).to_string())

def _sig_symbol(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''

fig, ax = plt.subplots(figsize=(13, 7))

x = np.arange(len(coefs_plot)) + 0.5
y = coefs_plot['Estimate'].values
yerr = coefs_plot['SE'].values
stars = [_sig_symbol(p) for p in coefs_plot['P-val'].values]

ax.errorbar(x, y, yerr=yerr, fmt='o', color=dot_color, ecolor=dot_color,
            elinewidth=2, capsize=4, markersize=10)
ax.axhline(0, linestyle='--', color='#333333', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(coefs_plot['Pretty'], rotation=45, ha='right', fontsize=14)
ax.set_ylabel('Effect Size \nPredicting Confidence at t', fontsize=16, fontweight='bold')
ax.tick_params(axis='y', labelsize=14)
ax.margins(y=0.15)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

for xi, yi, ei, s in zip(x, y, yerr, stars):
    if s:
        ax.text(xi, yi + ei + 0.5, s, ha='center', va='bottom',
                fontsize=18, color='black', fontweight='bold')

plt.tight_layout()
plt.show()
